# VSA-LM: Beat FlashLM SOTA on TinyStories

**Architecture**: BlockCode embeddings + AdditionLinear (no matmul) + LiquidCell recurrence + MindForge adaptive LoRA + Capsule cognitive features + Hippocampal recall

**Target**: PPL < 1.36 (FlashLM v5 Thunderbolt, 29.7M params)

**Runtime**: T4/L4 GPU via Colab

---

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q datasets tokenizers numpy loguru

# Clone CubeMind
!git clone https://github.com/grillcheese-ai/cubemind.git 2>/dev/null || (cd cubemind && git pull)
import sys; sys.path.insert(0, 'cubemind')

## 2. Prepare Data (ByteLevel BPE, vocab=8192, same as FlashLM)

In [ ]:
import os
import numpy as np
from loguru import logger

DATA_DIR = "vsa_lm_data"
VOCAB_SIZE = 8192
MAX_STORIES = 0  # 0 = all (~2.1M stories)

if not os.path.exists(os.path.join(DATA_DIR, "tokens.npy")):
    from datasets import load_dataset
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

    logger.info("Loading TinyStories...")
    ds = load_dataset("roneneldan/TinyStories", split="train")
    texts = [item["text"] for i, item in enumerate(ds) if MAX_STORIES == 0 or i < MAX_STORIES]
    logger.info(f"Loaded {len(texts)} stories")

    os.makedirs(DATA_DIR, exist_ok=True)

    # ByteLevel BPE — same as FlashLM v5
    logger.info(f"Training ByteLevel BPE (vocab={VOCAB_SIZE})...")
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
        min_frequency=2,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)
    tokenizer.save(os.path.join(DATA_DIR, "tokenizer.json"))

    # Tokenize
    logger.info("Tokenizing...")
    all_tokens = []
    for i, t in enumerate(texts):
        all_tokens.extend(tokenizer.encode(t).ids)
        if (i + 1) % 100000 == 0:
            logger.info(f"  {i+1}/{len(texts)} stories, {len(all_tokens)} tokens")

    tokens = np.array(all_tokens, dtype=np.int32)
    np.save(os.path.join(DATA_DIR, "tokens.npy"), tokens)
    np.save(os.path.join(DATA_DIR, "vocab_size.npy"), np.array([tokenizer.get_vocab_size()]))
    logger.info(f"Done: {len(tokens)} tokens ({tokens.nbytes/1e6:.0f}MB)")
else:
    logger.info("Data already prepared, loading...")
    tokens = np.load(os.path.join(DATA_DIR, "tokens.npy"))
    logger.info(f"Loaded {len(tokens)} tokens")

## 3. Load Data into Train/Val Sequences

In [ ]:
SEQ_LEN = 256  # Match FlashLM

vocab_size = int(np.load(os.path.join(DATA_DIR, "vocab_size.npy"))[0])
n = len(tokens)
train_tok = tokens[:int(0.8 * n)]
val_tok = tokens[int(0.8 * n):int(0.9 * n)]

def make_seqs(tok, seq_len):
    x, y = [], []
    for i in range(0, len(tok) - seq_len - 1, seq_len // 2):
        x.append(tok[i:i + seq_len])
        y.append(tok[i + 1:i + seq_len + 1])
    return np.array(x, dtype=np.int32), np.array(y, dtype=np.int32)

train_x, train_y = make_seqs(train_tok, SEQ_LEN)
val_x, val_y = make_seqs(val_tok, SEQ_LEN)
print(f"Vocab: {vocab_size}, Train: {len(train_x)} seqs, Val: {len(val_x)} seqs")

## 4. VSA-LM Model Definition

In [ ]:
import math
import time

# ── Capsule Constants ──
CAPSULE_DIM = 32
CAP_NOVELTY = 5
CAP_PLASTICITY = 14
CAP_CONSOLIDATION = 18


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=axis, keepdims=True) + 1e-8)


class LiquidCell:
    """Continuous-time recurrence with input-dependent time constants."""
    def __init__(self, d, dt=0.02, tau_min=0.02, tau_max=2.0, seed=42):
        rng = np.random.default_rng(seed)
        s = np.sqrt(2.0 / (d + d))
        self.W = rng.normal(0, s, (d, d)).astype(np.float32)
        self.U = rng.normal(0, s, (d, d)).astype(np.float32)
        self.b = np.zeros(d, dtype=np.float32)
        self.V = rng.normal(0, s, (d, d)).astype(np.float32)
        self.c = rng.normal(0, 0.1, (d,)).astype(np.float32)
        self.h = np.zeros(d, dtype=np.float32)
        self.dt = dt
        self.tau_min = tau_min
        self.tau_max = tau_max
        self.d = d

    def step(self, x):
        x = x.astype(np.float32).ravel()[:self.d]
        vx = self.V @ x + self.c
        tau = self.tau_min + np.log1p(np.exp(-np.abs(vx))) + np.maximum(vx, 0)
        tau = np.minimum(tau, self.tau_max)
        a = np.tanh(self.W @ self.h + self.U @ x + self.b)
        dh = -self.h / np.maximum(tau, 1e-6) + a
        self.h = (self.h + self.dt * dh).astype(np.float32)
        return self.h.copy()

    def param_count(self):
        return self.W.size + self.U.size + self.b.size + self.V.size + self.c.size


class AdditionLinear:
    """Multiplication-free linear: output = -||W - x||_1 + bias."""
    def __init__(self, d_in, d_out, bias=True, seed=42):
        rng = np.random.default_rng(seed)
        self.weight = rng.uniform(-0.1, 0.1, (d_out, d_in)).astype(np.float32)
        self.bias = np.zeros(d_out, dtype=np.float32) if bias else None
        self.in_features = d_in
        self.out_features = d_out

    def forward(self, x):
        # x: (S, d_in) → (S, d_out)
        if x.ndim == 1:
            x = x.reshape(1, -1)
        out = np.zeros((x.shape[0], self.out_features), dtype=np.float32)
        for i in range(self.out_features):
            out[:, i] = -np.abs(self.weight[i] - x).sum(axis=-1)
        if self.bias is not None:
            out += self.bias
        return out


class SignActivation:
    def forward(self, x):
        return np.sign(x).astype(np.float32)


class VSALayer:
    """LiquidCell recurrence + AdditionLinear FFN + capsule modulation."""
    def __init__(self, d, d_ffn, seed=42):
        self.ffn_up = AdditionLinear(d, d_ffn, bias=True, seed=seed)
        self.ffn_down = AdditionLinear(d_ffn, d, bias=True, seed=seed+1)
        self.sign = SignActivation()
        self.liquid = LiquidCell(d, seed=seed+2)
        self.ln_g = np.ones(d, dtype=np.float32)
        self.ln_b = np.zeros(d, dtype=np.float32)
        self.d = d

    def layernorm(self, x):
        m = x.mean(axis=-1, keepdims=True)
        v = x.var(axis=-1, keepdims=True)
        return self.ln_g * (x - m) / np.sqrt(v + 1e-5) + self.ln_b

    def forward(self, x, plasticity=0.5, consolidation=0.5):
        S, d = x.shape
        h = self.layernorm(x)
        x_mean = h.mean(axis=0)

        # LiquidCell with capsule modulation
        self.liquid.dt = 0.01 + 0.03 * plasticity
        self.liquid.tau_min = 0.02 + 0.08 * consolidation
        temporal = self.liquid.step(x_mean)

        # AdditionLinear FFN (no matmul)
        h_up = self.sign.forward(self.ffn_up.forward(h) / np.sqrt(self.ffn_up.in_features))
        h_ffn = self.ffn_down.forward(h_up) / np.sqrt(self.ffn_down.in_features)

        # Temporal gating
        gate = np.tanh(temporal)  # (d,)
        y = (0.5 + plasticity) * h_ffn * (1.0 + 0.1 * gate)
        return x + y

    def param_count(self):
        n = self.ffn_up.weight.size + self.ffn_down.weight.size
        if self.ffn_up.bias is not None: n += self.ffn_up.bias.size
        if self.ffn_down.bias is not None: n += self.ffn_down.bias.size
        n += self.ln_g.size + self.ln_b.size + self.liquid.param_count()
        return n


class VSALM:
    """VSA Language Model with capsule embeddings + LiquidCell recurrence."""
    def __init__(self, vocab, d, d_ffn, n_layers, seq_len, seed=42):
        rng = np.random.default_rng(seed)
        self.d = d
        self.n_layers = n_layers
        self.vocab = vocab
        self.seq_len = seq_len

        # Embeddings
        self.embed = rng.normal(0, 0.02, (vocab, d)).astype(np.float32)
        self.capsule_embed = rng.normal(0, 0.01, (vocab, CAPSULE_DIM)).astype(np.float32)
        self.capsule_proj = rng.normal(0, 0.02, (d, CAPSULE_DIM)).astype(np.float32)

        # Position encoding
        pe = np.zeros((seq_len + 16, d), dtype=np.float32)
        pos = np.arange(seq_len + 16).reshape(-1, 1).astype(np.float32)
        div = np.exp(np.arange(0, d, 2).astype(np.float32) * -(math.log(10000) / d))
        pe[:, 0::2] = np.sin(pos * div)
        pe[:, 1::2] = np.cos(pos * div[:d // 2])
        self.pe = pe

        # Output projection
        self.out_w = rng.normal(0, 0.02, (vocab, d)).astype(np.float32)

        # Layers
        self.layers = [VSALayer(d, d_ffn, seed=seed+l) for l in range(n_layers)]
        self._last_hidden = None

    def forward(self, ids, prev_loss=0.0):
        S = len(ids)
        x = self.embed[ids] + self.pe[:S]
        caps = self.capsule_embed[ids]
        x = x + (caps @ self.capsule_proj.T).astype(np.float32)

        caps_mean = caps.mean(axis=0)
        plasticity = float(np.clip(caps_mean[CAP_PLASTICITY], 0, 1))
        consolidation = float(np.clip(caps_mean[CAP_CONSOLIDATION], 0, 1))

        for layer in self.layers:
            x = layer.forward(x, plasticity=plasticity, consolidation=consolidation)

        self._last_hidden = x.copy()
        return (x @ self.out_w.T / np.sqrt(self.d)).astype(np.float32)

    def param_count(self):
        n = self.embed.size + self.capsule_embed.size + self.capsule_proj.size
        n += self.pe.size + self.out_w.size
        for l in self.layers:
            n += l.param_count()
        return n


print("Model defined.")

## 5. Initialize Model

In [ ]:
# Match FlashLM: d=384, 18 layers, d_ffn=1152
# Start smaller for validation, scale up after
D_MODEL = 384
N_LAYERS = 12
D_FFN = 1152
LR = 5e-4

model = VSALM(vocab_size, D_MODEL, D_FFN, N_LAYERS, SEQ_LEN, seed=42)
print(f"VSA-LM: d={D_MODEL}, layers={N_LAYERS}, vocab={vocab_size}, params={model.param_count()/1e6:.1f}M")

# Sanity check
test_logits = model.forward(train_x[0])
print(f"Forward OK: {test_logits.shape}, finite={np.all(np.isfinite(test_logits))}, std={test_logits.std():.2f}")

## 6. Training Loop

In [ ]:
TRAIN_STEPS = 50000
VAL_EVERY = 200

def compute_ppl(model, x_data, y_data, max_samples=100):
    total_loss, n = 0.0, 0
    for i in range(min(len(x_data), max_samples)):
        logits = model.forward(x_data[i])
        p = softmax(logits)
        for t in range(len(y_data[i])):
            tok = int(y_data[i][t])
            if 0 <= tok < p.shape[1]:
                total_loss -= math.log(max(float(p[t, tok]), 1e-8))
                n += 1
    return float(math.exp(min(total_loss / max(n, 1), 20)))


t0 = time.time()
best_ppl = float("inf")
step = 0
loss = 0.0

while step < TRAIN_STEPS:
    perm = np.random.permutation(len(train_x))
    train_x_s, train_y_s = train_x[perm], train_y[perm]

    for i in range(len(train_x_s)):
        if step >= TRAIN_STEPS:
            break

        ids = train_x_s[i]
        labels = train_y_s[i].ravel()
        S = len(ids)

        # Forward
        prev_loss = loss
        logits = model.forward(ids, prev_loss=prev_loss)
        x_final = model._last_hidden

        # CE loss
        probs = softmax(logits)
        loss = -np.mean(np.log(probs[np.arange(S), labels] + 1e-8))
        grad_logits = probs.copy()
        grad_logits[np.arange(S), labels] -= 1.0
        grad_logits /= S

        # Update output projection
        grad_out_w = grad_logits.T @ x_final / S
        model.out_w -= LR * np.clip(grad_out_w, -0.1, 0.1)

        # Update embeddings
        dx = (grad_logits @ model.out_w).astype(np.float32)
        for t in range(S):
            tok = int(ids[t])
            if 0 <= tok < model.vocab:
                model.embed[tok] -= LR * np.clip(dx[t], -0.1, 0.1)
                dcaps = (dx[t] @ model.capsule_proj).astype(np.float32)
                model.capsule_embed[tok] -= LR * np.clip(dcaps, -0.1, 0.1)

        # Update AdditionLinear weights
        error_signal = dx.mean(axis=0)
        for layer in model.layers:
            h_mean = x_final.mean(axis=0)[:layer.ffn_up.in_features]
            delta = np.outer(error_signal[:layer.ffn_up.out_features],
                             h_mean[:layer.ffn_up.in_features])
            if delta.shape == layer.ffn_up.weight.shape:
                layer.ffn_up.weight -= LR * 0.1 * np.clip(delta, -0.05, 0.05)

        # Log
        if step % VAL_EVERY == 0:
            val_ppl = compute_ppl(model, val_x, val_y, max_samples=50)
            elapsed = time.time() - t0
            sps = (step + 1) / elapsed if elapsed > 0 else 0
            print(f"step={step:6d} | loss={loss:.3f} | PPL={val_ppl:.1f} | {sps:.1f} stp/s")
            if val_ppl < best_ppl:
                best_ppl = val_ppl

        step += 1

print(f"\nDone: {step} steps, best PPL={best_ppl:.1f}, time={time.time()-t0:.0f}s")

## 7. Results

In [ ]:
final_ppl = compute_ppl(model, val_x, val_y, max_samples=200)
print(f"\n{'='*50}")
print(f"  VSA-LM Results")
print(f"{'='*50}")
print(f"  Final PPL:    {final_ppl:.2f}")
print(f"  Best PPL:     {best_ppl:.2f}")
print(f"  FlashLM SOTA: 1.36")
print(f"  Params:       {model.param_count()/1e6:.1f}M")
print(f"  Architecture: AdditionLinear + LiquidCell + Capsules")
print(f"  Training:     Error-driven (no full backprop)")
print(f"{'='*50}")

## 8. Save Checkpoint

In [ ]:
save_dict = {
    "embed": model.embed,
    "capsule_embed": model.capsule_embed,
    "capsule_proj": model.capsule_proj,
    "out_w": model.out_w,
    "best_ppl": np.array([best_ppl]),
    "step": np.array([step]),
}
for l, layer in enumerate(model.layers):
    save_dict[f"L{l}_ffn_up"] = layer.ffn_up.weight
    save_dict[f"L{l}_ffn_down"] = layer.ffn_down.weight
    save_dict[f"L{l}_ln_g"] = layer.ln_g
    save_dict[f"L{l}_ln_b"] = layer.ln_b
    save_dict[f"L{l}_liquid_W"] = layer.liquid.W
    save_dict[f"L{l}_liquid_U"] = layer.liquid.U
    save_dict[f"L{l}_liquid_V"] = layer.liquid.V
    save_dict[f"L{l}_liquid_b"] = layer.liquid.b
    save_dict[f"L{l}_liquid_c"] = layer.liquid.c

np.savez_compressed("vsa_lm_checkpoint.npz", **save_dict)
print(f"Saved checkpoint ({os.path.getsize('vsa_lm_checkpoint.npz')/1e6:.1f}MB)")